In [ ]:
import copy
import importlib
import re

import numpy as np
import pandas as pd
import matplotlib as mpl
import pyranges as pr
from scipy.spatial import cKDTree
import os

workdir = "/vast/ai_projects/Papenfuss_SVEnsemble2/model"
os.chdir(workdir)
os.getcwd()

import svensemble.vcfparse as vcfp 
import svensemble.bndbpconv as bndc
import svensemble.utils as sv_utils
import svensemble.caller_specific as caller_specific

importlib.reload(vcfp)
importlib.reload(bndc)
importlib.reload(sv_utils)
importlib.reload(caller_specific)

In [2]:
# Options and global vars
pd.set_option('display.max_columns', None)

caller_name = 'gridss'
sample_frac = 1
viable_svtypes = ['DEL', 'DUP', 'INS', 'INV']
evidence_margins = {
    'SR': 50,
    'PE': 100,
    'RD': 500
}


The current idea is:

- split the reference events into separate breakends (DELs have 1 breakpoint with 2 breakends, do others later)
- for each breakend, find the closest breakend in the gridss callset
- calculate the overlap fraction
- keep only those with > 80% overlap
- train model to predic correct breakends
- assemble into simple events later (before benchmark)


In [ ]:
if 'gridss_callset' not in locals() or 'gridss_meta' not in locals():
    gridss_callset, gridss_meta = vcfp.parse_vcf(
        "data/test_gridss_output.NA12878.vcf",
        line_limit = None, 
        output_meta=True
    )

sample_size_gridss = int(gridss_callset.shape[0] * sample_frac)
gridss_callset_sample = gridss_callset.sample(sample_size_gridss)
gridss_meta_sample = copy.deepcopy(gridss_meta)


In [ ]:
if 'ref_callset' not in locals() or 'ref_meta' not in locals():
    ref_callset, ref_meta = vcfp.parse_vcf(
        'data/gnomad.v4.1.sv.sites.vcf.gz',
        line_limit = None, skip_format=True, 
        info_limit_to=['SVTYPE', 'SVLEN', 'END', 'CHR2', 'EVIDENCE'],
        output_meta=True
    )

sample_size_ref = int(ref_callset.shape[0] * sample_frac)
ref_callset_sample = ref_callset.sample(sample_size_ref)
ref_meta_sample = copy.deepcopy(ref_meta)

In [ ]:
ref_callset_bps = caller_specific.callset_to_bp(ref_callset_sample, ref_meta_sample, source='gnomad')
gridss_callset_bps = caller_specific.callset_to_bp(gridss_callset_sample, gridss_meta_sample, source='gridss')

In [17]:
# def get_bnd_prs(bp_vcf_df, padding='INFO_CIPOS', suffixes=None, retain_cols=None):
#     """Extract breakend coordinates from a breakpoint VCF dataframe.
    
#     Takes a VCF dataframe in breakpoint notation and extracts the coordinates for both 
#     breakends, returning them as PyRanges objects. Can handle different padding fields
#     and retain additional columns from the input.

#     Args:
#         bp_vcf_df (pd.DataFrame): DataFrame containing breakpoint VCF data
#         padding (str or tuple): Either:
#             - Name of padding field in VCF (e.g. 'INFO_CIPOS') to use for coordinate calculation
#             - Tuple of (left_padding, right_padding) integers specifying how many bases to extend 
#               from each breakpoint position in both directions
#             Defaults to 'INFO_CIPOS'.
#         suffixes (tuple, optional): Tuple of suffixes used to identify first and second breakends.
#             Defaults to ('_bnd1', '_bnd2').
#         retain_cols (list, optional): Additional columns to retain from input DataFrame.
#             Defaults to None.

#     Returns:
#         tuple: (bnd1_coords, bnd2_coords) - Two PyRanges objects containing coordinates
#             for first and second breakends respectively.

#     Raises:
#         ValueError: If padding argument is not a string or tuple of integers.
#     """
#     if suffixes is None:
#         suffixes = ('_bnd1', '_bnd2')
    
#     if isinstance(padding, tuple):
#         if not all(isinstance(x, int) for x in padding):
#             raise ValueError('When padding is a tuple, it must contain two integers')
#         padding_cols_bnd1 = padding
#         padding_cols_bnd2 = padding
#     elif isinstance(padding, str):
#         padding_cols_bnd1 = tuple([bp_vcf_df[f'{padding}_{side}_{suffixes[0]}'] for side in ('l', 'r')])
#         padding_cols_bnd2 = tuple([bp_vcf_df[f'{padding}_{side}_{suffixes[1]}'] for side in ('l', 'r')])
#     else:
#         raise ValueError('padding must be a string or tuple of integers')
    
#     if retain_cols is None:
#         retain_cols = []

#     bnd1_coords = pd.DataFrame()
#     bnd2_coords = pd.DataFrame()

#     bnd1_suffix = suffixes[0] 
#     bnd2_suffix = suffixes[1]

#     # Extract first breakend coordinates
#     bnd1_coords.loc[:, 'Chromosome'] = bp_vcf_df[f'CHROM{bnd1_suffix}']
#     bnd1_coords.loc[:, 'Start'] = (bp_vcf_df[f'POS{bnd1_suffix}'] - 1 + 
#                                   padding_cols_bnd1[0])
#     bnd1_coords.loc[:, 'End'] = (bp_vcf_df[f'POS{bnd1_suffix}'] + 
#                                 padding_cols_bnd1[1])
#     bnd1_coords.loc[:, 'ID'] = bp_vcf_df['INFO_EVENT']
#     for col in retain_cols:
#         if not col.endswith(bnd2_suffix):
#             bnd1_coords.loc[:, col] = bp_vcf_df[col]

#     # Extract second breakend coordinates  
#     bnd2_coords.loc[:, 'Chromosome'] = bp_vcf_df[f'CHROM{bnd2_suffix}']
#     bnd2_coords.loc[:, 'Start'] = (bp_vcf_df[f'POS{bnd2_suffix}'] - 1 + 
#                                   padding_cols_bnd2[0])
#     bnd2_coords.loc[:, 'End'] = (bp_vcf_df[f'POS{bnd2_suffix}'] + 
#                                 padding_cols_bnd2[1])
#     bnd2_coords.loc[:, 'ID'] = bp_vcf_df['INFO_EVENT']
#     for col in retain_cols:
#         if not col.endswith(bnd1_suffix):
#             bnd2_coords.loc[:, col] = bp_vcf_df[col]

#     return pr.PyRanges(bnd1_coords), pr.PyRanges(bnd2_coords)



# def find_matching_breakpoint_ids(vcf_df1, vcf_df2):
#     """
#     Find IDs of matching breakpoints between two VCF dataframes.
    
#     Args:
#         vcf_df1, vcf_df2: VCF dataframes in breakpoint notation with CIPOS fields
#     Returns:
#         Array of IDs from vcf_df1 that have matching breakpoints in vcf_df2
#     """
#     # Get pyranges objects for both breakends of each VCF
#     bnd1_pr1, bnd2_pr1 = get_bnd_prs(vcf_df1, retain_cols=['bptype'])
#     bnd1_pr2, bnd2_pr2 = get_bnd_prs(vcf_df2, retain_cols=['bptype'])

#     # Find overlapping regions between corresponding breakends
#     joined_bnd1 = bnd1_pr1.join(bnd1_pr2, how=None, report_overlap=True).df
#     joined_bnd2 = bnd2_pr1.join(bnd2_pr2, how=None, report_overlap=True).df
    
#     # Calculate overlap fraction relative to both interval lengths
#     joined_bnd1.loc[:, 'overlap_frac'] = np.maximum(
#         joined_bnd1['Overlap'] / (joined_bnd1['End'] - joined_bnd1['Start']),
#         joined_bnd1['Overlap'] / (joined_bnd1['End_b'] - joined_bnd1['Start_b'])
#     )
#     joined_bnd2.loc[:, 'overlap_frac'] = np.maximum(
#         joined_bnd2['Overlap'] / (joined_bnd2['End'] - joined_bnd2['Start']),
#         joined_bnd2['Overlap'] / (joined_bnd2['End_b'] - joined_bnd2['Start_b'])
#     )

#     # Get IDs with >80% overlap and matching bptypes
#     bnd1_ids = joined_bnd1[
#         (joined_bnd1['overlap_frac'] > 0.8) &\
#         (joined_bnd1['bptype'] == joined_bnd1['bptype_b'])
#     ]['ID'].unique()
#     bnd2_ids = joined_bnd2[
#         (joined_bnd2['overlap_frac'] > 0.8) &\
#         (joined_bnd2['bptype'] == joined_bnd2['bptype_b'])
#     ]['ID'].unique()

#     # Return IDs matching in both breakends
#     return bnd1_ids[np.isin(bnd1_ids, bnd2_ids)]


In [18]:
# cipos_cols = ['INFO_CIPOS_l_bnd1', 'INFO_CIPOS_r_bnd1', 'INFO_CIPOS_l_bnd2', 'INFO_CIPOS_r_bnd2']

# # Get breakend coordinates for GRIDSS callset
# gridss_bnd1_pr, gridss_bnd2_pr = get_bnd_prs(gridss_callset_bps, padding=(0, 0), retain_cols=['bptype'] + cipos_cols)

# # Get breakend coordinates for reference callset
# ref_bnd1_pr, ref_bnd2_pr = get_bnd_prs(ref_callset_bps, padding=(0, 0), retain_cols=['bptype'] + cipos_cols)


In [6]:
gridss_callset_bps = bndc.sort_bnd_df(gridss_callset_bps, chrom_keys=sv_utils.generate_chrom_key_dict(pd.concat([gridss_callset_bps['CHROM_bnd1'], gridss_callset_bps['CHROM_bnd2']])))
ref_callset_bps = bndc.sort_bnd_df(ref_callset_bps, chrom_keys=sv_utils.generate_chrom_key_dict(pd.concat([ref_callset_bps['CHROM_bnd1'], ref_callset_bps['CHROM_bnd2']])))

In [39]:
def preprocess_bp_df_for_kdtree(bp_df):
    return bndc.sort_bnd_df(bp_df, chrom_keys=sv_utils.generate_chrom_key_dict(pd.concat([bp_df['CHROM_bnd1'], bp_df['CHROM_bnd2']])))

def get_chrom_bp_type(bp_df):
    """
    Utility function to get a unique identifier for chromosome position of a breakpoint + its type
    Requires a breakpoint dataframe
    """
    return bp_df['CHROM_bnd1'] + '_' + bp_df['CHROM_bnd2'] + '_' + bp_df['bp_type']

def get_kdtforest(bp_df, unique_chrom_bp_types):
    return {
        chrom_bp_type: cKDTree(bp_df[bp_df['chrom_bp_type'] == chrom_bp_type][['POS_bnd1', 'POS_bnd2']].to_numpy())
        for chrom_bp_type in unique_chrom_bp_types
    }

def kdtforest_query(kdtforest, bp_df_query, unique_chrom_bp_types_ref):
    return {
        chrom_bp_type: kdtforest[chrom_bp_type].query(bp_df_query[bp_df_query['chrom_bp_type'] == chrom_bp_type][['POS_bnd1', 'POS_bnd2']].to_numpy())
        for chrom_bp_type in unique_chrom_bp_types_ref
    }

def match_nearest_query(
    bp_df_query,
    bp_df_ref,
    ref_closest,
    unique_chrom_bp_types_query,
    chrom_bp_types_ref,
    caller_name='defaultcaller'
):
    query_matches = pd.DataFrame()
    ref_matches = pd.DataFrame()

    unmatched_chrom_bp_types = []
    for chrom_bp_type in unique_chrom_bp_types_query:
        # Filter reference breakpoints by chromosomes and breakpoint type
        filtered_bp_df_query = bp_df_query[
            bp_df_query['chrom_bp_type'] == chrom_bp_type
        ]
        filtered_bp_df_ref = bp_df_ref[
            bp_df_ref['chrom_bp_type'] == chrom_bp_type
        ]
        
        if filtered_bp_df_ref.shape[0] == 0:
            unmatched_chrom_bp_types.append(chrom_bp_type)
            continue
            
        # Reset index to ensure proper indexing when selecting matches
        filtered_bp_df_ref = filtered_bp_df_ref.reset_index(drop=True)
        
        # Get indices of closest reference breakpoints
        matched_indices = ref_closest[chrom_bp_type][1]
        
        # Select matching reference breakpoints and add to results
        matched_refs = filtered_bp_df_ref.iloc[matched_indices]
        ref_matches = pd.concat([ref_matches, matched_refs])
        query_matches = pd.concat([query_matches, filtered_bp_df_query])

    ref_matches = ref_matches.reset_index(drop=True).add_suffix('_ref')
    query_matches = query_matches.reset_index(drop=True).add_suffix(f'_{caller_name}')

    unmatched_calls = gridss_callset_bps[gridss_callset_bps['chrom_bp_type'].isin(unmatched_chrom_bp_types)].reset_index(drop=True).add_suffix(f'_{caller_name}')

    return pd.concat([pd.concat([query_matches, ref_matches], axis=1), unmatched_calls], axis=0).reset_index(drop=True)


def match_nearest_breakpoints(
    bp_df_query,
    bp_df_ref,
    k=1
):
    bp_df_query_preprocessed = preprocess_bp_df_for_kdtree(bp_df_query)
    bp_df_ref_preprocessed = preprocess_bp_df_for_kdtree(bp_df_ref)

    chrom_bp_types_ref = get_chrom_bp_type(bp_df_ref_preprocessed)
    chrom_bp_types_query = get_chrom_bp_type(bp_df_query_preprocessed)

    bp_df_query_preprocessed.loc[:, 'chrom_bp_type'] = chrom_bp_types_query
    bp_df_ref_preprocessed.loc[:, 'chrom_bp_type'] = chrom_bp_types_ref

    unique_chrom_bp_types_ref = chrom_bp_types_ref.unique()
    unique_chrom_bp_types_query = chrom_bp_types_query.unique()

    kdtforest = get_kdtforest(bp_df_ref_preprocessed, unique_chrom_bp_types_ref)

    ref_nearest_to_query = kdtforest_query(kdtforest, bp_df_query_preprocessed, unique_chrom_bp_types_ref)

    return match_nearest_query(bp_df_query_preprocessed, bp_df_ref_preprocessed, ref_nearest_to_query, unique_chrom_bp_types_query, unique_chrom_bp_types_ref)

    

In [40]:
gridss_ref_matched = match_nearest_breakpoints(gridss_callset_bps, ref_callset_bps)

In [ ]:
gridss_ref_matched

In [12]:
bnd1_left_boundary = matched_total['POS_bnd1_ref'] + matched_total['INFO_CIPOS_l_bnd1_ref'] + matched_total[f'INFO_CIPOS_l_bnd1_{caller_name}']
bnd1_right_boundary = matched_total['POS_bnd1_ref'] + matched_total['INFO_CIPOS_r_bnd1_ref'] + matched_total[f'INFO_CIPOS_r_bnd1_{caller_name}']
bnd2_left_boundary = matched_total['POS_bnd2_ref'] + matched_total['INFO_CIPOS_l_bnd2_ref'] + matched_total[f'INFO_CIPOS_l_bnd2_{caller_name}']
bnd2_right_boundary = matched_total['POS_bnd2_ref'] + matched_total['INFO_CIPOS_r_bnd2_ref'] + matched_total[f'INFO_CIPOS_r_bnd2_{caller_name}']

ref_match_bnd1_mask = (matched_total[f'POS_bnd1_{caller_name}'] >= bnd1_left_boundary) & (matched_total[f'POS_bnd1_{caller_name}'] <= bnd1_right_boundary)
ref_match_bnd2_mask = (matched_total[f'POS_bnd2_{caller_name}'] >= bnd2_left_boundary) & (matched_total[f'POS_bnd2_{caller_name}'] <= bnd2_right_boundary)

matched_total.loc[:, 'ref_match'] = ref_match_bnd1_mask & ref_match_bnd2_mask

In [ ]:
matched_total['ref_match'].value_counts()

In [ ]:
# Checking if there is significant difference in QUAL between ref_match and not ref_match

import matplotlib.pyplot as plt
from scipy import stats

def create_qual_boxplot(data, qual_col, ax, ylim=10000):
    """Create boxplot and calculate statistics for a QUAL column"""
    data.boxplot(column=qual_col, by='ref_match', ax=ax)
    ax.set_ylim(0, ylim)
    
    stat, pval = stats.mannwhitneyu(
        data[data['ref_match']][qual_col],
        data[~data['ref_match']][qual_col]
    )
    ax.set_title(f'{qual_col}\np={pval:.2e}')

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 8))

# Create boxplots for each QUAL column
qual_columns = ['QUAL_bnd1_gridss', 'QUAL_bnd2_gridss', 
                'QUAL_bnd1_ref', 'QUAL_bnd2_ref']
axes = [ax1, ax2, ax3, ax4]

for qual_col, ax in zip(qual_columns, axes):
    create_qual_boxplot(matched_total, qual_col, ax)

plt.tight_layout()
plt.savefig('figs/misc/qual_vs_refmatch.png')


In [ ]:
matched_total_ref_match_only = matched_total[matched_total['ref_match']]
matched_total_ref_match_only


In [ ]:
gridss_callset_bps[~gridss_callset_bps['chrom_bp_type'].isin(unmatched_chrom_bptypes)]

In [ ]:
gridss_callset_bps[gridss_callset_bps['chrom_bptype'].isin(ref_matches['chrom_bptype'].unique())]


In [ ]:
gridss_ref_closest = pd.concat([gridss_callset_bps[gridss_callset_bps['chrom_bptype'].isin(ref_matches['chrom_bptype'].unique())].reset_index(drop=True), ref_matches.reset_index(drop=True).add_suffix('_ref')], axis=1)
gridss_ref_closest.loc[:, 'eucl_dist'] = np.sqrt((gridss_ref_closest['POS_bnd1'] - gridss_ref_closest['POS_bnd1_ref'])**2 + (gridss_ref_closest['POS_bnd2'] - gridss_ref_closest['POS_bnd2_ref'])**2)
gridss_ref_closest.loc[:, 'manhattan_dist'] = np.abs(gridss_ref_closest['POS_bnd1'] - gridss_ref_closest['POS_bnd1_ref']) + np.abs(gridss_ref_closest['POS_bnd2'] - gridss_ref_closest['POS_bnd2_ref'])


# gridss_ref_closest.loc[:, 'dist_cutoff'] = np.sqrt(
#     (
#         (-gridss_ref_closest['INFO_CIPOS_l_bnd1'] + gridss_ref_closest['INFO_CIPOS_r_bnd1']) / 2 
#         + 100000 
#         - gridss_ref_closest['INFO_CIPOS_l_bnd1_ref']
#     ) ** 2 
#     + (
#         (gridss_ref_closest['INFO_CIPOS_l_bnd2'] + gridss_ref_closest['INFO_CIPOS_r_bnd2']) / 2
#         + 100000 
#         - gridss_ref_closest['INFO_CIPOS_l_bnd2_ref']
#     ) ** 2
# )

gridss_ref_closest.loc[:, 'is_close'] = gridss_ref_closest['manhattan_dist'] < 1000
gridss_ref_closest_test = gridss_ref_closest[gridss_ref_closest['is_close']][['CHROM_bnd1', 'POS_bnd1', 'CHROM_bnd2', 'POS_bnd2', 'CHROM_bnd1_ref', 'POS_bnd1_ref', 'CHROM_bnd2_ref', 'POS_bnd2_ref', 'manhattan_dist', 'INFO_EVENT', 'ALT_bnd1', 'INFO_EVENT_ref']]
gridss_ref_closest_test

In [ ]:
ref_callset_bps[ref_callset_bps['chrom_bptype'] == 'chr2_chr2_21'].iloc[ref_closes_to_gridss['chr2_chr2_21'][1]]

In [ ]:
gridss_callset_bps

In [ ]:
set(gridss_callset_bps['bptype']).union(set(ref_callset_bps['bptype']))

In [ ]:
gridss_callset_bps.shape

In [ ]:
callset_chrom_combs = set(pd.concat([(gridss_callset_bps['CHROM_bnd1'] + '_' + gridss_callset_bps['CHROM_bnd2']), (ref_callset_bps['CHROM_bnd1'] + '_' + ref_callset_bps['CHROM_bnd2'])]))
callset_chrom_combs

In [ ]:
wall_dist = 10000

unique_bptypes = set(gridss_callset_bps['bptype']).union(set(ref_callset_bps['bptype']))

distance_df = pd.DataFrame()

for bptype in unique_bptypes:
    gridss_callset_bp_bybptype = gridss_callset_bps[gridss_callset_bps['bptype'] == bptype]
    ref_callset_bp_bybptype = ref_callset_bps[ref_callset_bps['bptype'] == bptype]
    if gridss_callset_bp_bybptype.shape[0] == 0 or ref_callset_bp_bybptype.shape[0] == 0:
        print(f'No breakpoints of type {bptype} in one of the callsets, skipping matching')
        continue
    # Get breakend coordinates for GRIDSS callset
    gridss_bnd1_bybptype_pr, gridss_bnd2_bybptype_pr = get_bnd_prs(gridss_callset_bp_bybptype, padding=(0, 0), retain_cols=['bptype'])

    # Get breakend coordinates for reference callset
    ref_bnd1_bybptype_pr, ref_bnd2_bybptype_pr = get_bnd_prs(ref_callset_bp_bybptype, padding=(0, 0), retain_cols=['bptype'])

    gridss_bnd1_bybptype_pr_kn = gridss_bnd1_bybptype_pr.k_nearest(ref_bnd1_bybptype_pr, k=1).df
    gridss_bnd1_dist_to_same_type_ref = gridss_bnd1_bybptype_pr_kn[['ID', 'Start', 'Start_b', 'Distance', 'ID_b']]
    gridss_bnd1_dist_to_same_type_ref = gridss_bnd1_dist_to_same_type_ref.rename(columns={'Distance': 'INFO_BND1_DIST', 'Start': 'Start_bnd1', 'Start_b': 'Start_b_bnd1', 'ID_b': 'ID_b_bnd1'})
    gridss_bnd2_bybptype_pr_kn = gridss_bnd2_bybptype_pr.k_nearest(ref_bnd2_bybptype_pr, k=1).df
    gridss_bnd2_dist_to_same_type_ref = gridss_bnd2_bybptype_pr_kn[['Start', 'Start_b', 'Distance', 'ID_b']]
    gridss_bnd2_dist_to_same_type_ref = gridss_bnd2_dist_to_same_type_ref.rename(columns={'Distance': 'INFO_BND2_DIST', 'Start': 'Start_bnd2', 'Start_b': 'Start_b_bnd2', 'ID_b': 'ID_b_bnd2'})

    gridss_dist_bybptype = pd.concat([gridss_bnd1_dist_to_same_type_ref, gridss_bnd2_dist_to_same_type_ref], axis=1)

    gridss_unmatched_mask = (np.abs(gridss_dist_bybptype['INFO_BND1_DIST']) > wall_dist) | (np.abs(gridss_dist_bybptype['INFO_BND2_DIST']) > wall_dist)

    distance_df = pd.concat([distance_df, gridss_dist_bybptype])

print(distance_df)




In [ ]:
bp_array1 = gridss_callset_bps[['POS_bnd1', 'POS_bnd2']].to_numpy()
bp_array2 = ref_callset_bps[['POS_bnd1', 'POS_bnd2']].to_numpy()

tree = cKDTree(bp_array2)
dists, idx = tree.query(bp_array1, 1)
# nn_dist = dists[0][:, 1]

dists.shape

In [136]:
from scipy.spatial import cKDTree


In [ ]:
for chrom1 in gridss_callset_bps['CHROM_bnd1'].unique():
    for chrom2 in gridss_callset_bps['CHROM_bnd1'].unique():
        mask1 = (gridss_callset_bps['CHROM_bnd1'] == chrom1) & (gridss_callset_bps['CHROM_bnd2'] == chrom2)
        mask2 = (ref_callset_bps['CHROM_bnd1'] == chrom1) & (ref_callset_bps['CHROM_bnd2'] == chrom2)
        # print(gridss_callset_bps[mask1])
        # print(ref_callset_bps[mask2])

        bp_array1 = gridss_callset_bps[mask1][['POS_bnd1', 'POS_bnd2']].to_numpy()
        bp_array2 = ref_callset_bps[mask2][['POS_bnd1', 'POS_bnd2']].to_numpy()

        if bp_array1.shape[0] == 0 or bp_array2.shape[0] == 0:
            continue
        
        tree = cKDTree(bp_array2)
        dists, idx = tree.query(bp_array1, 1)
        print(idx)

        matched_gridss_bnds = gridss_callset_bps[mask1].reset_index(drop=True)
        matched_ref_bnds = ref_callset_bps[mask2].iloc[idx].reset_index(drop=True)

        matched_bnds = pd.concat([matched_gridss_bnds, matched_ref_bnds], axis=1)
        print(matched_bnds)

matched_bnds


In [ ]:
bp_array2.shape

In [ ]:
gridss_callset_bps

In [ ]:
distance_df[distance_df['ID_b_bnd1'] == distance_df['ID_b_bnd2']]

In [ ]:
        gridss_dist_bybptype[full_matches_mask]['ID_b_bnd1']

        gridss_callset_bp_bybptype = gridss_callset_bp_bybptype[gridss_callset_bp_bybptype['ID'].isin(ids_fully_unmatched)]
        ref_callset_bp_bybptype = ref_callset_bp_bybptype[~ref_callset_bp_bybptype['ID'].isin(matched_ids_b)]

        distance_df = pd.concat([distance_df, gridss_dist_bybptype[full_matches_mask]])




In [ ]:
no_matches_mask = (np.abs(distance_df['INFO_BND1_DIST']) > wall_dist) | (np.abs(distance_df['INFO_BND2_DIST']) > wall_dist)
full_matches_mask = ~no_matches_mask & (distance_df['ID_b_bnd1'] == distance_df['ID_b_bnd2'])

distance_df[~(no_matches_mask | full_matches_mask)]


In [ ]:
gridss_callset_bps[~gridss_callset_bps['INFO_EVENT'].isin(distance_df['ID'])]

In [ ]:
gridss_bnd1_pr_kn = gridss_bnd1_pr.k_nearest(ref_bnd1_pr, k=1).df
# gridss_bnd1_pr_kn = gridss_bnd1_pr_kn[(gridss_bnd1_pr_kn['Distance'] == 0) & (gridss_bnd1_pr_kn['bptype'] == gridss_bnd1_pr_kn['bptype_b'])]
gridss_bnd1_pr_kn = gridss_bnd1_pr_kn[(gridss_bnd1_pr_kn['bptype'] == gridss_bnd1_pr_kn['bptype_b'])]
gridss_bnd1_pr_kn

In [ ]:
# Join GRIDSS and reference coordinates to find overlapping regions
# 'left' join keeps all GRIDSS calls and matches them to reference where possible
# report_overlap=True adds an 'Overlap' column with the overlap length
joined_call_ref_bnd1_pr = gridss_bnd1_pr.join(ref_bnd1_pr, how='left', report_overlap=True).df
joined_call_ref_bnd2_pr = gridss_bnd2_pr.join(ref_bnd2_pr, how='left', report_overlap=True).df

# Calculate overlap fraction for first breakend coordinates
# Takes maximum of:
# 1. Overlap length divided by GRIDSS interval length
# 2. Overlap length divided by reference interval length
# This accounts for cases where one interval contains the other
joined_call_ref_bnd1_pr.loc[:, 'overlap_frac'] = np.maximum(
    joined_call_ref_bnd1_pr['Overlap'] / (joined_call_ref_bnd1_pr['End'] - joined_call_ref_bnd1_pr['Start']),
    joined_call_ref_bnd1_pr['Overlap'] / (joined_call_ref_bnd1_pr['End_b'] - joined_call_ref_bnd1_pr['Start_b'])
)

# Calculate overlap fraction for second breakend coordinates
# Uses same approach as first breakend
joined_call_ref_bnd2_pr.loc[:, 'overlap_frac'] = np.maximum(
    joined_call_ref_bnd2_pr['Overlap'] / (joined_call_ref_bnd2_pr['End'] - joined_call_ref_bnd2_pr['Start']),
    joined_call_ref_bnd2_pr['Overlap'] / (joined_call_ref_bnd2_pr['End_b'] - joined_call_ref_bnd2_pr['Start_b'])
)

# Find IDs where first breakend has good overlap with reference
# Requires:
# 1. >80% overlap between GRIDSS and reference coordinates
# 2. Matching breakend types between GRIDSS and reference
bnd1_overlapped_ids = joined_call_ref_bnd1_pr[
    (joined_call_ref_bnd1_pr['overlap_frac'] > 0.8) &\
    (joined_call_ref_bnd1_pr['bptype'] == joined_call_ref_bnd1_pr['bptype_b'])
]['ID'].unique()

# Find IDs where second breakend has good overlap with reference
# Uses same criteria as first breakend
bnd2_overlapped_ids = joined_call_ref_bnd2_pr[
    (joined_call_ref_bnd2_pr['overlap_frac'] > 0.8) &\
    (joined_call_ref_bnd2_pr['bptype'] == joined_call_ref_bnd2_pr['bptype_b'])
]['ID'].unique()

# Return arrays of IDs that have good overlap in both breakends
bnd1_overlapped_ids, bnd2_overlapped_ids


In [ ]:
bnd12_overlapped_ids = bnd1_overlapped_ids[np.isin(bnd1_overlapped_ids, bnd2_overlapped_ids)]
bnd12_overlapped_ids


In [ ]:
matching_ids = find_matching_breakpoint_ids(gridss_callset_bps, ref_callset_bps)
matching_ids


In [ ]:
gridss_callset_bps.loc[:, 'is_in_training'] = gridss_callset_bps['INFO_EVENT'].isin(matching_ids)
gridss_callset_bps.is_in_training.value_counts()

In [38]:
# Columns to remove from the final dataset
# cols_to_drop_approx = [
#     # Location columns
#     'CHROM', 'POS', 'Chromosome', 'Start', 'End',
    
#     # Reference/variant columns 
#     'REF', 'ALT',
    
#     # Format columns
#     'FORMAT_GT', 'FORMAT_VF',
    
#     # INFO fields with multiple values/string values
#     'INFO_SVLEN',
#     'INFO_ASC',
#     'INFO_BEID', 
#     'INFO_EVENT',
#     'INFO_HOMSEQ',
#     'INFO_MATEID',
#     'INFO_SC',
#     'INFO_SVTYPE',
#     'INFO_CIPOS',
#     'INFO_CIRPOS',
    
#     # Encoding column
#     'bnd_pos_encoding'
# ]

gridss_callset_bps_mult_val_cols = gridss_callset_bps.nunique() > 10
gridss_callset_bps_one_val_cols = gridss_callset_bps.nunique() == 1
gridss_callset_bps_obj_cols = gridss_callset_bps.dtypes == 'object'

gridss_callset_bps_cols_to_drop = gridss_callset_bps.columns[(gridss_callset_bps_mult_val_cols & gridss_callset_bps_obj_cols) | gridss_callset_bps_one_val_cols]

gridss_force_include = ['INFO_EVENT', 'bptype']

gridss_callset_bps_cols_to_drop = gridss_callset_bps_cols_to_drop[~gridss_callset_bps_cols_to_drop.isin(gridss_force_include)]

# cols_to_drop = []

# Create final dataset by dropping unnecessary columns
gridss_callset_final = gridss_callset_bps.drop(columns=gridss_callset_bps_cols_to_drop)

In [ ]:
gridss_callset_final

In [ ]:
gridss_callset_final.to_csv('saved/gridss.overlapped_bps.csv', index=False)

In [10]:

# ref_dels_bp_start = ref_callset_dels[['CHROM', 'POS', 'ID']]
# ref_dels_bp_start = ref_dels_bp_start.rename(columns={'CHROM': 'Chromosome'})
# ref_dels_bp_start.loc[:, 'Start'] = ref_dels_bp_start['POS'] - 100
# ref_dels_bp_start.loc[:, 'End'] = ref_dels_bp_start['POS'] + 101

# ref_dels_bp_end = ref_callset_dels[['INFO_CHR2', 'INFO_END', 'ID']]
# ref_dels_bp_end = ref_dels_bp_end.rename(columns={'INFO_CHR2': 'Chromosome'})
# ref_dels_bp_end.loc[:, 'Start'] = ref_dels_bp_end['INFO_END'] - 100
# ref_dels_bp_end.loc[:, 'End'] = ref_dels_bp_end['INFO_END'] + 101


In [11]:
# ref_dels_bp_start_pr = pr.PyRanges(ref_dels_bp_start)
# ref_dels_bp_end_pr = pr.PyRanges(ref_dels_bp_end)

In [31]:
# gridss_call_test = gridss_callset

# grsizes = gridss_call_test.groupby('INFO_EVENT').transform('size')
# gridss_call_test = gridss_call_test[grsizes == 2].sort_values(['INFO_EVENT', 'CHROM', 'POS'])
# gridss_call_test_start, gridss_call_test_end = gridss_call_test.iloc[::2].reset_index(drop=True), gridss_call_test.iloc[1::2].reset_index(drop=True)

In [32]:

# gridss_call_test_start = gridss_call_test_start[['CHROM', 'POS', 'ID']]
# gridss_call_test_start = gridss_call_test_start.rename(columns={'CHROM': 'Chromosome'})
# gridss_call_test_start.loc[:, 'Start'] = gridss_call_test_start['POS'] - 100
# gridss_call_test_start.loc[:, 'End'] = gridss_call_test_start['POS'] + 101

# gridss_call_test_end = gridss_call_test_end[['CHROM', 'POS', 'ID']]
# gridss_call_test_end = gridss_call_test_end.rename(columns={'CHROM': 'Chromosome'})
# gridss_call_test_end.loc[:, 'Start'] = gridss_call_test_end['POS'] - 100 
# gridss_call_test_end.loc[:, 'End'] = gridss_call_test_end['POS'] + 101


# gridss_call_test_end_pr = pr.PyRanges(gridss_call_test_end)
# gridss_call_test_start_pr = pr.PyRanges(gridss_call_test_start)

In [33]:
# gridss_call_test_start_pr = gridss_call_test_start_pr.join(ref_dels_bp_start_pr, how='left', report_overlap=True).df.reset_index(drop=True)
# gridss_call_test_start_pr.loc[:, 'overlap_frac'] = 2 * gridss_call_test_start_pr['Overlap'] / (
#     gridss_call_test_start_pr['End'] - gridss_call_test_start_pr['Start'] + gridss_call_test_start_pr['End_b'] - gridss_call_test_start_pr['Start_b']
# )

# # gridss_call_test_start_pr.loc[:, 'overlap_frac'].value_counts()

In [ ]:
# gridss_call_test_start_pr.sort_values(['ID', 'overlap_frac'], ascending=[True, False]).drop_duplicates(subset='ID', keep='first')[gridss_call_test_start_pr['overlap_frac'] > 0.8]